# Figure 7. Residual decay for the two directional systems

Reproduces Figure 7 (\S5.2.2) of *A Spectrally Damped Tensor Randomized
Kaczmarz Method for Doubly Noisy Tensor Systems*.

- **Experiment 5.2.2** — row-directional and column-directional residual evolution for
  standard TRK vs. SD-TRK, on the same Geisel Library deblurring run as Figure 6
  (`image_residual_decay.png`)

**Common setting.** Identical experiment to Figure 6 (\S5.2.2): $\mathcal{X}^{\mathrm{true}}$ is the
Geisel Library image, resized to $256\times170\times3$; row/column Gaussian blur operators
(standard deviation $2.0$, truncation radius $8$) are embedded as t-product operators with
$\delta_A=10^{-4}$ operator perturbation and $\delta_B=10^{-2}$ observation noise, pixel values
clipped to $[0,1]$. Recovery proceeds in two directional passes — row system
$\tilde{\mathcal{A}}^{r}\mathcal{X}=\tilde{\mathcal{B}}$, then column system
$\tilde{\mathcal{A}}^{c}\mathcal{Y}=(\mathcal{X}^{\mathrm{row}})^{\top_{12}}$ — each initialized
from its degraded right-hand side. **SD-TRK** uses $\omega=0.30$, $\lambda=10^{-2}$; the
**standard TRK** baseline is the same solver with $\omega=1$, $\lambda=0$; both run $1200$
iterations per pass, same seed (`41`) and sampling sequence as Figure 6.

Here we additionally record the observable residual norm
$\|\tilde{\mathcal{A}}^{r}\mathcal{X}^t-\tilde{\mathcal{B}}\|_F$ (row pass) and
$\|\tilde{\mathcal{A}}^{c}\mathcal{Y}^t-(\mathcal{X}^{\mathrm{row}})^{\top_{12}}\|_F$ (column pass)
at every iteration for both methods.

**Expected result.** Residual trajectories stabilize near nonzero plateaus (the observed
right-hand sides contain image noise and are clipped after noise is added, so driving the
residual to zero is not aligned with recovering the clean image). In the row pass, both methods
decrease during the early iterations; in the column pass, the standard TRK residual initially
increases slightly before stabilizing.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
def load_image(path, maxside):
    # RGB image resized so max(height, width) = maxside, normalized to [0, 1]
    img = Image.open(path).convert('RGB')
    scale = maxside / max(img.size)
    if scale < 1.0:
        img = img.resize((max(1, round(img.size[0] * scale)),
                          max(1, round(img.size[1] * scale))), Image.Resampling.LANCZOS)
    return np.asarray(img, dtype=np.float64) / 255.0


def blur_matrix(n, sigma, radius):
    # 1D Gaussian blur matrix with reflexive boundary conditions
    offsets = np.arange(-radius, radius + 1)
    kernel = np.exp(-offsets**2 / (2.0 * sigma**2))
    kernel /= kernel.sum()
    M = np.zeros((n, n))
    for i in range(n):
        for off, w in zip(offsets, kernel):
            j = i + int(off)
            if j < 0:
                j = -j
            if j >= n:
                j = 2 * n - j - 2
            M[i, j] += w
    return M


def embed(M, n3=3):
    # embed a matrix as a t-product operator: first frontal slice = M, rest zero
    A = np.zeros((*M.shape, n3))
    A[:, :, 0] = M
    return A


def t_prod(A, B):
    # t-product: FFT along mode 3 -> slice-wise matmul -> inverse FFT (Appendix A)
    n1, _, n3 = A.shape
    n4 = B.shape[1]
    A_hat = np.fft.fft(A, axis=2)
    B_hat = np.fft.fft(B, axis=2)
    C_hat = np.zeros((n1, n4, n3), dtype=complex)
    for k in range(n3):
        C_hat[:, :, k] = A_hat[:, :, k] @ B_hat[:, :, k]
    return np.fft.ifft(C_hat, axis=2).real


def t12(Z):
    # Z^T12: transpose each frontal slice (swaps image row/column directions only,
    # NOT the t-product conjugate transpose)
    return np.transpose(Z, (1, 0, 2))


def relerr(Z, ref):
    return np.linalg.norm(Z - ref) / np.linalg.norm(ref)


def psnr(Z, ref):
    mse = np.mean((np.clip(Z, 0.0, 1.0) - ref)**2)
    return 10.0 * np.log10(1.0 / mse)


def sdtrk_solve_history(Aop, B, X0, omega, lam, seed):
    # same update as Figure 6's sdtrk_solve, additionally recording the residual
    # norm ||Aop X^t - B||_F at every iteration (bookkeeping only; does not draw
    # from rng, so the returned reconstruction is bit-identical to Figure 6)
    n1, _, n3 = Aop.shape
    A_hat = np.fft.fft(Aop, axis=2)
    B_hat = np.fft.fft(B, axis=2)
    X_hat = np.fft.fft(X0, axis=2)
    rowsq = np.array([np.linalg.norm(Aop[i])**2 for i in range(n1)])
    prob = rowsq / rowsq.sum()
    rng = np.random.default_rng(seed)
    history = []
    for t in range(T):
        i = int(rng.choice(n1, p=prob))
        for k in range(n3):
            a = A_hat[i, :, k]
            denom = np.vdot(a, a).real + lam
            resid = a @ X_hat[:, :, k] - B_hat[i, :, k]
            X_hat[:, :, k] -= omega * np.outer(a.conj(), resid) / denom
        X_current = np.fft.ifft(X_hat, axis=2).real
        history.append(np.linalg.norm(t_prod(Aop, X_current) - B))
    Xfinal = np.clip(np.fft.ifft(X_hat, axis=2).real, 0.0, 1.0)
    return Xfinal, history

In [ ]:
maxside = 256            # processed image: 256 x 170 x 3
sigma, radius = 2.0, 8   # Gaussian blur std and truncation radius
delta_A, delta_B = 1e-4, 1e-2
T = 1200                 # iterations per directional pass
omega, lam = 0.30, 1e-2  # SD-TRK relaxation and scalar damping
seed = 41


def two_pass_with_history(om, lm):
    # row-direction pass, then column-direction pass; X^rec = Y^T12
    # each pass starts from its degraded observation; residual history is
    # recorded per pass in the pass's own directional system
    Xrow, row_hist = sdtrk_solve_history(Ar, Btilde, Btilde, om, lm, seed + 100)
    Y, col_hist = sdtrk_solve_history(Ac, t12(Xrow), t12(Xrow), om, lm, seed + 101)
    return t12(Y), row_hist, col_hist

In [ ]:
Xtrue = load_image(Path('geisel.jpg'), maxside)
height, width, _ = Xtrue.shape
print('processed size:', f'{width} x {height} x 3')

rng = np.random.default_rng(seed)
Armat = blur_matrix(height, sigma, radius)
Acmat = blur_matrix(width, sigma, radius)
Ar = embed(Armat + delta_A * rng.standard_normal(Armat.shape))  # observed A~^r for the solvers
Ac = embed(Acmat + delta_A * rng.standard_normal(Acmat.shape))  # observed A~^c for the solvers

# blur generated with the noise-free operators, then observation noise + clipping
Bblur = t12(t_prod(embed(Acmat), t12(t_prod(embed(Armat), Xtrue))))
Btilde = np.clip(Bblur + delta_B * rng.standard_normal(Bblur.shape), 0.0, 1.0)

Xtrk, trk_row_hist, trk_col_hist = two_pass_with_history(1.0, 0.0)     # standard TRK baseline
Xrec, sd_row_hist, sd_col_hist = two_pass_with_history(omega, lam)     # SD-TRK

# sanity check against Figure 6's reconstruction metrics (same seed and rng calls)
print(f"{'Image':<28} {'Rel. error':>10} {'PSNR (dB)':>10}")
for name, Z in [('Degraded observation', Btilde), ('Standard TRK', Xtrk), ('SD-TRK', Xrec)]:
    print(f'{name:<28} {relerr(Z, Xtrue):>10.4f} {psnr(Z, Xtrue):>10.4f}')

print()
print(f"{'Pass':<12} {'Standard TRK':>14} {'SD-TRK':>14}  (final residual)")
print(f"{'Row':<12} {trk_row_hist[-1]:>14.4f} {sd_row_hist[-1]:>14.4f}")
print(f"{'Column':<12} {trk_col_hist[-1]:>14.4f} {sd_col_hist[-1]:>14.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
steps = np.arange(1, T + 1)

axes[0].plot(steps, trk_row_hist, color='black', linestyle='--', linewidth=1.2, label='Standard TRK')
axes[0].plot(steps, sd_row_hist, color='black', linewidth=1.5, label='SD-TRK')
axes[0].set_title('Row-directional system')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel(r'Residual norm $\|\tilde{\mathcal{A}}\mathcal{X}^t-\tilde{\mathcal{B}}\|_F$')
axes[0].legend(frameon=False)

axes[1].plot(steps, trk_col_hist, color='black', linestyle='--', linewidth=1.2, label='Standard TRK')
axes[1].plot(steps, sd_col_hist, color='black', linewidth=1.5, label='SD-TRK')
axes[1].set_title('Column-directional system')
axes[1].set_xlabel('Iteration')
axes[1].legend(frameon=False)

plt.savefig('image_residual_decay.png', dpi=300, bbox_inches='tight')
plt.show()